In [78]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pinocchio as pin
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from pynumdiff.optimize import optimize
from pynumdiff.polynomial_fit import savgoldiff
from pynumdiff.finite_difference import finitediff
from pynumdiff.smooth_finite_difference import butterdiff
from scipy.spatial.transform import Rotation as R
from pinocchio.visualize import MeshcatVisualizer
import meshcat.transformations as tf
import requests
from flask import Flask, request, jsonify, render_template_string
from threading import Thread
import webbrowser
import time

In [79]:
def load_datapoints(path, n_components, names):
    df = pd.read_csv(path)

    # One row per component, so group every n_components rows
    values = df["value"].to_numpy().reshape(-1, n_components)

    # Use the first row of each vector for its timestamp
    times = df["elapsed time"].to_numpy()[::n_components]

    result = pd.DataFrame(values, columns=names)
    result["time"] = times

    return result

In [80]:
def load_joints(path, names):
    data = pd.read_csv(path)

    #Each message contains 20 rows; the first 12 are joint values.
    n_messages = len(data) // 20
    blocks = data["value"].to_numpy()[:n_messages * 20].reshape(-1, 20)
    time_blocks = data["elapsed time"].to_numpy()[:n_messages * 20].reshape(-1, 20)

    result = pd.DataFrame(blocks[:, :12], columns=names)
    result["time"] = time_blocks[:, 0]
    return result

In [81]:
def savgol_derivative(omega, dt):
    _, acceleration = savgoldiff(omega, dt, **savgol_params_value) 
    return acceleration

In [82]:
def finite_derivative(omega, dt):
    _, finite_acceleration = finitediff(omega, dt, **finite_params_value)
    return finite_acceleration

In [83]:
def butter_derivative(omega, dt):
    _, butter_acceleration = butterdiff(omega, dt, **butter_params_value)
    return butter_acceleration

In [84]:
def build_corridor_dataframe():

    import os
    import numpy as np
    import pandas as pd

    motorstate = "./motorstate/"
    highstate = "./highstate/"

    leg_order = [
        "frhip", "frthigh", "frcalf",
        "flhip", "flthigh", "flcalf",
        "rlhip", "rlthigh", "rlcalf",
        "rrhip", "rrthigh", "rrcalf"
    ]

    df = pd.DataFrame()

    q_file = os.path.join(motorstate, "q_data/corridor_q.csv")
    dq_file = os.path.join(motorstate, "dq_data/corridor_dq.csv")
    ddq_file = os.path.join(motorstate, "ddq_data/corridor_ddq.csv")


    def load_joints(path, names):
        data = pd.read_csv(path)

        #Each message contains 20 rows; the first 12 are joint values.
        n_messages = len(data) // 20
        blocks = data["value"].to_numpy()[:n_messages * 20].reshape(-1, 20)
        time_blocks = data["elapsed time"].to_numpy()[:n_messages * 20].reshape(-1, 20)

        result = pd.DataFrame(blocks[:, :12], columns=names)
        result["time"] = time_blocks[:, 0]
        return result

    q_joint = load_joints(q_file, leg_order)
    dq_joint = load_joints(dq_file, leg_order)
    ddq_joint = load_joints(ddq_file, leg_order)

    for i in range(12):
        df[f"q{i}"] = q_joint.iloc[:, i]
        df[f"v{i}"] = dq_joint.iloc[:, i]
        df[f"a{i}"] = ddq_joint.iloc[:, i]

    #Load highstate data
    velocity_file = os.path.join(highstate, "corridor_velocity_x.csv")
    roll_file = os.path.join(highstate, "corridor_roll.csv")
    pitch_file = os.path.join(highstate, "corridor_pitch.csv")

    df['velocity'] = pd.read_csv(velocity_file).iloc[:, -1]
    df['roll'] = pd.read_csv(roll_file).iloc[:, -1]
    df['pitch'] = pd.read_csv(pitch_file).iloc[:, -1]

    df_clean = df.dropna(subset=['velocity', 'roll', 'pitch'])

    #Drop samples from data overlap (update frequency was 50 Hz not 500)
    df_cleanest = df_clean.iloc[::10].copy().reset_index(drop=True)

    df_cleanest['z'] = 0.3106581568717956
    return leg_order, df, q_file, dq_file, ddq_file, df_clean, df_cleanest

In [85]:
def load_go1_model():
    notebook_dir = Path.cwd()
    urdf_go1 = (notebook_dir / "../../models/unitree_go1.urdf").resolve() 
    models_dir = urdf_go1.parent

    go1, collision_model, visual_model = pin.buildModelsFromUrdf(
        str(urdf_go1), 
        package_dirs=[
            str(models_dir.parent),
            str(models_dir)
        ], 
        root_joint=pin.JointModelFreeFlyer()
    )
    return go1, collision_model, visual_model

In [86]:
def load_highstate_data():
    position = load_datapoints(
        "./highstate/data/corridor_position.csv",
        3,
        ["x", "y", "z"]
    )

    velocity = load_datapoints(
        "./highstate/data/corridor_velocity.csv",
        3,
        ["vx", "vy", "vz"]
    )

    gyro = load_datapoints(
        "./highstate/data/corridor_gyroscope.csv",
        3,
        ["wx", "wy", "wz"]
    )

    accel = load_datapoints(
        "./highstate/data/corridor_accelerometer.csv",
        3,
        ["ax", "ay", "az"]
    )

    quat = load_datapoints(
        "./highstate/data/corridor_quaternion.csv",
        4,
        ["qx", "qy", "qz", "qw"]
    )

    ff = load_datapoints(
        "./highstate/data/corridor_footforce.csv",
        4,
        ["FR", "FL", "RL", "RR"]
    )
    return position, velocity, gyro, accel, quat, ff

In [87]:
def build_pinocchio_state(q_file, leg_order, dq_file, ddq_file, position, quat, velocity, gyro):
    pin_idx = [3, 4, 5,
               0, 1, 2,
               6, 7, 8,
               9, 10, 11]

    q_joint = load_joints(q_file, leg_order)
    dq_joint = load_joints(dq_file, leg_order)
    ddq_joint = load_joints(ddq_file, leg_order)

    q_pin_joints = q_joint.iloc[:, pin_idx].copy()
    dq_pin_joints = dq_joint.iloc[:, pin_idx].copy()
    ddq_pin_joints = ddq_joint.iloc[:, pin_idx].copy()

    q_pin = np.column_stack([
        position[["x", "y", "z"]].to_numpy(),
        quat[["qx", "qy", "qz", "qw"]].to_numpy(),
        q_pin_joints
    ])

    q_pin[:, 3:7] /= np.linalg.norm(
        q_pin[:, 3:7], #qx, qy, qz, qw 
        axis=1,
        keepdims=True
    )

    v_pin = np.column_stack([
        velocity[["vx", "vy", "vz"]].to_numpy(),
        gyro[["wx", "wy", "wz"]].to_numpy(),
        dq_pin_joints
    ])
    return ddq_joint, q_pin, v_pin

In [88]:
def estimate_angular_acceleration():
    global butter_params_value, finite_params_value, savgol_params_value
    angular_vel_x = pd.read_csv("./imu_raw/corridor_angvel_x.csv")
    angular_vel_y = pd.read_csv("./imu_raw/corridor_angvel_y.csv")
    angular_vel_z = pd.read_csv("./imu_raw/corridor_angvel_z.csv")

    dt = 1/50

    omega_x = angular_vel_x["value"].to_numpy(dtype=float)[::10]
    omega_y = angular_vel_y["value"].to_numpy(dtype=float)[::10]
    omega_z = angular_vel_z["value"].to_numpy(dtype=float)[::10]

    #optimize once

    savgol_params_value, _ = optimize(
        savgoldiff,
        omega_x,
        dt,
        search_space_updates={
            "degree": 2,
            "window_size": [3, 10, 30, 50, 90, 130],
            "smoothing_win": [3, 10, 30, 50, 90, 130],
        },
    )

    finite_params_value, _ = optimize(
        finitediff, 
        omega_x, 
        dt)

    butter_params_value, _ = optimize(
        butterdiff,
        omega_x,
        dt,
        search_space_updates={
            "filter_order": (2),
            "cutoff_freq": [0.05, 0.1, 0.2],
            "num_iterations": [1, 2],
        },
        parallel=False,
    )

    savgol_alpha_x = savgol_derivative(omega_x, dt)
    savgol_alpha_y = savgol_derivative(omega_y, dt)
    savgol_alpha_z = savgol_derivative(omega_z, dt)

    finite_alpha_x = finite_derivative(omega_x, dt)
    finite_alpha_y = finite_derivative(omega_y, dt)
    finite_alpha_z = finite_derivative(omega_z, dt)

    butter_alpha_x = butter_derivative(omega_x, dt)
    butter_alpha_y = butter_derivative(omega_y, dt)
    butter_alpha_z = butter_derivative(omega_z, dt)
    return savgol_alpha_x, finite_alpha_x, butter_alpha_x, butter_alpha_y, butter_alpha_z

In [89]:
def plot_angular_acceleration(savgol_alpha_x, finite_alpha_x, butter_alpha_x):
    plt.figure(figsize=(14, 5))

    plt.plot(savgol_alpha_x, label="Savitzky–Golay",
             color="purple", alpha=1.0, linewidth=1, zorder=3)

    plt.plot(finite_alpha_x, label="Finite differences",
             color="cyan", alpha=0.6, linewidth=1, zorder=2)

    plt.plot(butter_alpha_x, label="Butterworth",
             color="yellow", alpha=0.7, linewidth=1, zorder=3)

    plt.xlabel("Sample")
    plt.ylabel("Angular acceleration (rad/s²)")
    plt.title("Angular acceleration estimates — X axis")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [90]:
def prepare_dynamics_inputs(accel, ddq_joint, butter_alpha_x, butter_alpha_y, butter_alpha_z):
    #Base IMU acceleration f_B from the accelerometer
    f_body = accel[["ax", "ay", "az"]].to_numpy(dtype=float)

    #Joint acceleration, reordered to Pinocchio's joint order
    ddq_pin_joints = ddq_joint.iloc[:, :12].to_numpy(dtype=float)

    #Base angular acceleration
    alpha = np.column_stack([
        butter_alpha_x,
        butter_alpha_y,
        butter_alpha_z,
    ])
    return f_body, ddq_pin_joints, alpha

In [91]:
def compute_body_acceleration(quat, f_body):
    q = quat[["qx", "qy", "qz", "qw"]].to_numpy(dtype=float)
    q = q / np.linalg.norm(q, axis=1, keepdims=True)

    rotation = R.from_quat(q)

    g_world = np.array([0.0, 0.0, -9.81]) #gravity

    g_body = rotation.inv().apply(g_world)

    a_body = f_body + g_body #our a_B should be f_B + g_B
    return rotation, a_body

In [92]:
def filter_body_acceleration(f_body, a_body):
    def lowpass(x, fs, fc, order=4):
        b, a = butter(order, fc / (fs / 2))
        return filtfilt(b, a, x, axis=0)

    #Rows where the accelerometer actually updated (true 50 Hz samples)
    changed = np.any(np.diff(f_body, axis=0) != 0, axis=1)
    upd_idx = np.r_[0, np.where(changed)[0] + 1]

    #Filter on the unique updates only, at the real 50 Hz rate
    a_upd_f = lowpass(a_body[upd_idx], fs=50.0, fc=8.0)

    # Hold each filtered value across the rows it covers, so indexing still works
    k_all = np.searchsorted(upd_idx, np.arange(len(a_body)), side="right") - 1
    a_body_filt = a_upd_f[k_all]
    return a_body_filt

In [93]:
def downsample_to_50hz(df_clean, df, q_pin, v_pin, a_body_filt, ddq_pin_joints, gyro, alpha):
    valid_indices = df_clean.index.to_numpy()

    sample_indices = np.intersect1d(
        np.arange(len(df))[::10],
        valid_indices
    )
    
    sample_indices = sample_indices[sample_indices // 10 < len(alpha)]

    q_pin_clean = q_pin[sample_indices]
    v_pin_clean = v_pin[sample_indices]

    a_body_50 = a_body_filt[sample_indices]
    ddq_pin_50 = ddq_pin_joints[sample_indices]
    gyro_50 = gyro.drop(columns=["time"]).iloc[sample_indices].to_numpy()

    #alpha is ALREADY 50 Hz
    alpha_50 = alpha[sample_indices // 10]
    return sample_indices, q_pin_clean, v_pin_clean, a_body_50, ddq_pin_50, gyro_50, alpha_50

In [94]:
def print_dropped_samples(df, df_clean):
    print("df:", len(df))
    print("df_clean:", len(df_clean))
    print("dropped:", len(df) - len(df_clean))

In [95]:
def compute_adjusted_body_acceleration(a_body_50, alpha_50, gyro_50, ddq_pin_50):
    #xyz="-0.01592 -0.06659 -0.00617"
    imu_pos = np.array([-0.01592,
                        -0.06659,
                        -0.00617])

    print(a_body_50.shape)
    print(alpha_50.shape)
    print(gyro_50.shape)
    print(imu_pos.shape)

    #Convert specific force into classical linear acceleration
    g_world = np.array([0.0, 0.0, -9.81])

    adjusted_a_body = (a_body_50 - np.cross(alpha_50, imu_pos) - np.cross(gyro_50, np.cross(gyro_50, imu_pos)))

    a_pin_clean = np.column_stack([
        adjusted_a_body,
        alpha_50,
        ddq_pin_50])
    return imu_pos, adjusted_a_body, a_pin_clean

In [96]:
def check_classical_acceleration(rotation, sample_indices, a_body, alpha_50, imu_pos, gyro_50, adjusted_a_body):
    g_world = np.array([0.0, 0.0, -9.81])

    R_WB_50 = rotation.as_matrix()[sample_indices]

    a_raw_50 = a_body[sample_indices]

    a_classical_body = (
        a_raw_50
        - np.cross(alpha_50, imu_pos)
        - np.cross(
            gyro_50,
            np.cross(gyro_50, imu_pos)
        )
    )

    print("reconstructed:")
    print(a_classical_body[-1])

    print("current adjusted:")
    print(adjusted_a_body[-1])

    print("difference:")
    print(a_classical_body[-1] - adjusted_a_body[-1])

In [97]:
def check_sample_lengths(df_cleanest, q_pin_clean):
    return len(df_cleanest), len(q_pin_clean)

In [98]:
def select_rnea_samples(df_cleanest, q_pin_clean, v_pin_clean, a_pin_clean):
    keep_idx = df_cleanest.index.to_numpy()

    q_pin_rnea = q_pin_clean[keep_idx]
    v_pin_rnea = v_pin_clean[keep_idx]
    a_pin_rnea = a_pin_clean[keep_idx]
    return q_pin_rnea, v_pin_rnea, a_pin_rnea

### We want Fx, Fy, Fz, tx, and ty. Use the RNEA which will return those results.

In [99]:
def compute_rnea_forces_moments(go1, q_pin_rnea, v_pin_rnea, a_pin_rnea, df_cleanest):
    fx_list, fy_list, fz_list = [], [], []
    mx_list, my_list = [], []

    data = go1.createData()

    for i in range(len(q_pin_rnea)):
        q = q_pin_rnea[i]
        v = v_pin_rnea[i]
        a = a_pin_rnea[i]

        tau = pin.rnea(go1, data, q, v, a)

        Fx, Fy, Fz = tau[:3]
        Mbase_x, Mbase_y, Mbase_z = tau[3:6]

        fx_list.append(Fx)
        fy_list.append(Fy)
        fz_list.append(Fz)

        z = df_cleanest["z"].iloc[i]
    
        #Mground_x = Mbase_x - Fx * z
        Mground_x = Mbase_x - (Fy * z)
        Mground_y = Mbase_y + (Fx * z)
    
        mx_list.append(Mground_x)
        my_list.append(Mground_y)   

    df_cleanest['Fx'] = fx_list
    df_cleanest['Fy'] = fy_list
    df_cleanest['Fz'] = fz_list
    df_cleanest['Mx'] = mx_list
    df_cleanest['My'] = my_list
    return df_cleanest

In [100]:
def print_mean_linear_acceleration(a_pin_clean):
    print(a_pin_clean[:, :3].mean(axis=0))

In [101]:
def describe_forces_moments(df_cleanest):
    return df_cleanest[['Fx', 'Fy', 'Fz', 'Mx', 'My']].describe()

In [102]:
def count_nan_fz(df_cleanest):
    print(df_cleanest["Fz"].isna().sum())

In [103]:
def print_mean_fz(df_cleanest):
    print(df_cleanest["Fz"].mean())

In [104]:
def compute_robot_weight(go1):
    m = pin.computeTotalMass(go1)
    print('mg =', m*9.81)
    return m

In [105]:
def compute_fz_zscore(df_cleanest):
    df_cleanest['z_score'] = ((df_cleanest['Fz']-128.159642)/59.309825)
    return df_cleanest, df_cleanest['z_score'].describe()

In [106]:
def count_fz_outliers(df_cleanest):
    print("Fz > 3 sd from mean:", (df_cleanest['Fz'] > (df_cleanest['Fz'].mean() + (3*df_cleanest["Fz"].std()))).sum())
    print("Fz < 3 sd from mean:", (df_cleanest['Fz'] < (df_cleanest['Fz'].mean() - (3*df_cleanest["Fz"].std()))).sum())
    print("Fz = 0:", (df_cleanest['Fz'] == 0).sum())

In [107]:
def flag_investigate_samples(df_cleanest):
    # Negative forces
    negative = df_cleanest["Fz"] < 0

    # Extreme standardized values
    extreme = df_cleanest["z_score"].abs() > 3

    print("Negative Fz:", negative.sum())
    print("Extreme z-scores:", extreme.sum())
    print("Both:", (negative & extreme).sum())
    print("Negative but not extreme:", (negative & ~extreme).sum())
    print("Extreme but nonnegative:", (~negative & extreme).sum())

    df_cleanest["negative_Fz"] = df_cleanest["Fz"] < 0
    df_cleanest["extreme_z"] = df_cleanest["z_score"].abs() > 3

    df_cleanest["investigate"] = (
        df_cleanest["negative_Fz"]
        | df_cleanest["extreme_z"]
    )
    return df_cleanest

In [108]:
def compute_neutral_foot_positions(go1):
    standing = pin.neutral(go1)
    data = go1.createData()

    pin.forwardKinematics(go1, data, standing)
    pin.updateFramePlacements(go1, data)

    feet = ["FR_foot", "FL_foot", "RL_foot", "RR_foot"]

    for foot in feet:
        #Find the frame ID inside the URDF
        frame_id = go1.getFrameId(foot)
    
        #Grab the 3D translations vector [X, Y, Z] from the center (0,0,0)
        foot_xyz = data.oMf[frame_id].translation
    
        print(
            f"{foot}: "
            f"x={foot_xyz[0]:.3f}, "
            f"y={foot_xyz[1]:.3f}, "
            f"z={foot_xyz[2]:.3f}"
        )
    
        #Extract directional distances
        forward_backward_dist = abs(foot_xyz[0])
        side_to_side_dist = abs(foot_xyz[1])
    
        print(f"{foot} Center Position:")
        print(f"Front-to-Back Distance (X): {forward_backward_dist:.3f} meters")
        print(f"Side-to-Side Distance  (Y): {side_to_side_dist:.3f} meters\n")
    return data, forward_backward_dist, side_to_side_dist

In [109]:
def compute_zmp_instability(forward_backward_dist, side_to_side_dist, df_cleanest):
    #Distance between center of robot and the center of the leg + radius of the foot.
    X_max = forward_backward_dist + 0.02
    Y_max = side_to_side_dist + 0.02

    df_cleanest["X_zmp"] = -df_cleanest["My"] / df_cleanest["Fz"]
    df_cleanest["Y_zmp"] =  df_cleanest["Mx"] / df_cleanest["Fz"]

    df_cleanest['instability_X'] = df_cleanest['X_zmp'].abs() / X_max
    df_cleanest['instability_Y'] = df_cleanest['Y_zmp'].abs() / Y_max

    df_cleanest['instability_index'] = df_cleanest[['instability_X', 'instability_Y']].max(axis=1)

    print(df_cleanest.head())
    df_cleanest[['velocity', 'roll', 'pitch', 'instability_index']].to_csv('corridor_instability.csv', index = False)
    return df_cleanest

In [110]:
def count_instability_outliers(df_cleanest):
    print("instability > 3 sd from mean:", (df_cleanest['instability_index'] > (df_cleanest['instability_index'].mean() + (3*df_cleanest["instability_index"].std()))).sum())
    print("instability < 3 sd from mean:", (df_cleanest['instability_index'] < (df_cleanest['instability_index'].mean() - (3*df_cleanest["instability_index"].std()))).sum())
    print("instability > threshold (1):", (df_cleanest['instability_index'] > 1).sum())

In [111]:
def compute_foot_positions(q_pin_rnea, go1, data):

    feet = ["FR_foot", "FL_foot", "RL_foot", "RR_foot"]
    foot_ids = {foot: go1.getFrameId(foot) for foot in feet}

    foot_xyz = np.zeros((len(q_pin_rnea), len(feet), 3))

    for i, q in enumerate(q_pin_rnea):
        pin.forwardKinematics(go1, data, q)
        pin.updateFramePlacements(go1, data)

        for j, foot in enumerate(feet):
            foot_xyz[i, j] = data.oMf[foot_ids[foot]].translation

    print("Foot positions shape:", foot_xyz.shape)
    print("First sample (FR, FL, RL, RR):")
    print(foot_xyz[0])
    return foot_xyz, i

In [112]:
def print_footforce_head(ff):
    print(ff.head(10000)) #Here, the robot looks to be stationary with a constant foot force.

In [113]:
def describe_footforce(ff):
    print(ff[['FR', 'FL', 'RL', 'RR']].head(3029).describe().loc[['mean', 'std']])


    print(ff[['FR', 'FL', 'RL', 'RR']].iloc[3030:6058].describe().loc[['mean', 'std']])

    print(ff[["FR", "FL", "RL", "RR"]].agg(["min", "max"]))

In [114]:
def extract_footforce_array(ff):
    forces=ff[["FR", "FL", "RL", "RR"]].to_numpy()
    return forces

In [115]:
def sweep_contact_thresholds(forces):
    thresholds = np.arange(10, 301, 10)

    results = []

    for threshold in thresholds:
        contacts = forces > threshold
        n_contacts = contacts.sum(axis=1)

        results.append({
            "threshold": threshold,
            "zero_contact": np.mean(n_contacts == 0),
            "one_contact": np.mean(n_contacts == 1),
            "two_contacts": np.mean(n_contacts == 2),
            "three_contacts": np.mean(n_contacts == 3),
            "four_contacts": np.mean(n_contacts == 4),
            "at_least_three": np.mean(n_contacts >= 3),
        })

    threshold_results = pd.DataFrame(results)
    return threshold_results

In [116]:
def plot_contact_threshold_sweep(threshold_results):
    import matplotlib.pyplot as plt

    plt.plot(
        threshold_results["threshold"],
        threshold_results["at_least_three"],
        marker="o"
    )
    plt.xlabel("Contact threshold")
    plt.ylabel("Fraction with at least 3 feet in contact")
    plt.grid(True)
    plt.show()

In [117]:
def compute_stance_mask(ff):
    contact_threshold = 100

    stance_mask = (
        ff[["FR", "FL", "RL", "RR"]].to_numpy() > contact_threshold
    )
    return stance_mask

In [118]:
def plot_support_polygon(go1, data, q_pin_rnea, i, foot_xyz, stance_mask, df_cleanest):
    pin.forwardKinematics(go1, data, q_pin_rnea[i])
    pin.updateFramePlacements(go1, data)

    i = 1000

    stance_xy = foot_xyz[i, stance_mask[i], :2]
    zmp_xy = df_cleanest.loc[i, ["X_zmp", "Y_zmp"]].to_numpy(dtype=float)

    fig, ax = plt.subplots()

    ax.scatter(foot_xyz[i, :, 0] - foot_xyz[i, :, 0].mean(),
               foot_xyz[i, :, 1] - foot_xyz[i, :, 1].mean(),
               alpha=0)  # no-op; coordinates below use the base frame

    base_xy = data.oMi[go1.getJointId("root_joint")].translation[:2]
    feet_xy = foot_xyz[i, :, :2] - base_xy
    stance_xy = feet_xy[stance_mask[i]]

    ax.scatter(feet_xy[:, 0], feet_xy[:, 1], label="Feet")
    ax.scatter(*zmp_xy, marker="x", s=100, label="ZMP")

    if len(stance_xy) >= 3:
        from scipy.spatial import ConvexHull

        hull = ConvexHull(stance_xy)
        polygon = np.vstack([
            stance_xy[hull.vertices],
            stance_xy[hull.vertices][0]
        ])
        ax.plot(polygon[:, 0], polygon[:, 1], label="Support polygon")

    ax.set_xlabel("X relative to base (m)")
    ax.set_ylabel("Y relative to base (m)")
    ax.set_aspect("equal")
    ax.legend()
    plt.show()

In [119]:
def init_meshcat_viewer(go1, collision_model, visual_model):
    import numpy as np
    import pinocchio as pin
    from pinocchio.visualize import MeshcatVisualizer
    import time

    #Re-instantiate a clean visualizer using the working models
    viz = MeshcatVisualizer(go1, collision_model, visual_model)

    #Re-attach to current viewer instance
    viz.initViewer(open=True)

    #Load the model
    viz.loadViewerModel(rootNodeName="pinocchio")
    return viz

In [120]:
def start_dashboard_server(viz):
    global latest
    app = Flask(__name__)
    meshcat_url = viz.viewer.url()

    HTML = """
<!doctype html>
<html>
<head>
    <title>Go1 ZMP Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
    <style>
        body {
            margin: 0;
            font-family: Arial, sans-serif;
        }
        header {
            padding: 12px;
            font-size: 20px;
        }
        .layout {
            display: grid;
            grid-template-columns: 1fr 1fr;
            height: calc(100vh - 50px);
            gap: 8px;
            padding: 8px;
            box-sizing: border-box;
        }
        .panel {
            min-width: 0;
            min-height: 0;
            overflow: hidden;
        }
        iframe, #plot {
            width: 100%;
            height: 100%;
            border: 0;
        }
    </style>
</head>
<body>
<header>Go1 Support Polygon and ZMP</header>

<div class="layout">
    <div class="panel">
        <iframe src="{{ meshcat_url }}"></iframe>
    </div>
    <div class="panel"><div id="plot"></div></div>
</div>

<script>
const layout = {
    title: "Go1 Support Polygon and ZMP",
    xaxis: {
        title: "X relative to base (m)",
        zeroline: true,
        range: [-1, 1],
        scaleanchor: "y",
        scaleratio: 1
    },
    yaxis: {
        title: "Y relative to base (m)",
        zeroline: true,
        range: [-1, 1]
    },
    margin: {l: 60, r: 20, t: 50, b: 60},
    legend: {orientation: "h", y: -0.2},
    uirevision: "go1"
};

const config = {
    responsive: true,
    displaylogo: false
};

Plotly.newPlot("plot", [
    {
        x: [], y: [],
        mode: "markers",
        name: "Feet",
        marker: {size: 10}
    },
    {
        x: [], y: [],
        mode: "markers",
        name: "ZMP",
        marker: {symbol: "x", size: 14}
    },
    {
        x: [], y: [],
        mode: "lines",
        name: "Support polygon",
        line: {width: 2}
    }
], layout, config);

function convexHull(points) {
    const pts = points
        .map(p => [Number(p[0]), Number(p[1])])
        .filter(p => p.every(Number.isFinite))
        .sort((a, b) => a[0] - b[0] || a[1] - b[1]);

    const cross = (o, a, b) =>
        (a[0] - o[0]) * (b[1] - o[1]) -
        (a[1] - o[1]) * (b[0] - o[0]);

    if (pts.length <= 1) return pts;

    function half(arr) {
        const h = [];
        for (const p of arr) {
            while (
                h.length >= 2 &&
                cross(h[h.length - 2], h[h.length - 1], p) <= 0
            ) {
                h.pop();
            }
            h.push(p);
        }
        return h.slice(0, -1);
    }

    return half(pts).concat(half([...pts].reverse()));
}

let polling = false;

async function refreshPlot() {
    if (polling) return;
    polling = true;

    try {
        const response = await fetch("/state", {cache: "no-store"});
        if (!response.ok) return;

        const s = await response.json();

        if (!Array.isArray(s.feet_xy) ||
            !Array.isArray(s.zmp_xy)) return;

        const feet = s.feet_xy
            .map(p => [Number(p[0]), Number(p[1])])
            .filter(p => p.every(Number.isFinite));

        const zmp = [Number(s.zmp_xy[0]), Number(s.zmp_xy[1])];

        if (!zmp.every(Number.isFinite)) return;

        const stance = Array.isArray(s.stance_xy) ? s.stance_xy : [];
        let polygon = [];

        if (stance.length >= 3) {
            polygon = convexHull(stance);
            if (polygon.length >= 3) polygon.push(polygon[0]);
        }

        await Plotly.restyle("plot", {
            x: [
                feet.map(p => p[0]),
                [zmp[0]],
                polygon.map(p => p[0])
            ],
            y: [
                feet.map(p => p[1]),
                [zmp[1]],
                polygon.map(p => p[1])
            ]
        }, [0, 1, 2]);

    } catch (e) {
        console.error("Plot refresh failed:", e);
    } finally {
        polling = false;
    }
}

refreshPlot();
setInterval(refreshPlot, 100);
</script>
</body>
</html>
"""

    latest = {}

    @app.route("/")
    def home():
        return render_template_string(HTML, meshcat_url=meshcat_url)

    @app.route("/state")
    def state():
        return jsonify(latest)

    @app.route("/update", methods=["POST"])
    def update():
        global latest
        latest = request.get_json(silent=True) or {}
        return jsonify({"ok": True})

    def run_server():
        app.run(
            host="127.0.0.1",
            port=5055,
            debug=False,
            use_reloader=False,
            threaded=True
        )

    Thread(target=run_server, daemon=True).start()
    time.sleep(1)
    webbrowser.open("http://127.0.0.1:5055")
    return app

In [121]:
def follow_camera(q, distance=2.0, height=1.2):
    x, y, z = q[:3]

    #Camera offset relative to the robot
    camera_pos = np.array([x - distance, y, z + height])

    #Point the camera toward the robot
    target = np.array([x, y, z + 0.2])

    #Build a camera pose looking toward the target
    direction = target - camera_pos
    direction /= np.linalg.norm(direction)

    up = np.array([0.0, 0.0, 1.0])
    right = np.cross(direction, up)
    right /= np.linalg.norm(right)
    up = np.cross(right, direction)

    rotation = np.column_stack((right, up, -direction))
    pose = np.eye(4)
    pose[:3, :3] = rotation
    pose[:3, 3] = camera_pos

    viz.viewer["/Cameras/default"].set_transform(pose)

In [122]:
def run_dashboard_playback(go1, df_cleanest, q_pin_rnea, foot_xyz, stance_mask, viz, data):
    dashboard_url = "http://127.0.0.1:5055/update"
    root_id = go1.getJointId("root_joint")

    x_zmp = df_cleanest['X_zmp']
    y_zmp = df_cleanest['Y_zmp']
    q_view = q_pin_rnea.copy()
    q_view[:, :3] -= q_view[0, :3]

    n = min(len(q_view), len(x_zmp), len(y_zmp), len(foot_xyz), len(stance_mask))

    for i, q in enumerate(q_view):
        viz.display(q)
        follow_camera(q)

        pin.forwardKinematics(go1, data, q_pin_rnea[i])
        pin.updateFramePlacements(go1, data)

        base_xy = data.oMi[root_id].translation[:2]

        feet_xy = foot_xyz[i, :, :2] - base_xy
        stance_xy = feet_xy[stance_mask[i]]

        zmp_xy = (
            df_cleanest.loc[i, ["X_zmp", "Y_zmp"]]
            .to_numpy(dtype=float)
            )

        requests.post(dashboard_url, json={
            "feet_xy": feet_xy.tolist(),
            "stance_xy": stance_xy.tolist(),
            "zmp_xy": zmp_xy.tolist()
        })

        time.sleep(0.05)

In [123]:
def generate_corridor_instability():
    leg_order, df, q_file, dq_file, ddq_file, df_clean, df_cleanest = build_corridor_dataframe()
    go1, collision_model, visual_model = load_go1_model()
    position, velocity, gyro, accel, quat, ff = load_highstate_data()
    ddq_joint, q_pin, v_pin = build_pinocchio_state(q_file, leg_order, dq_file, ddq_file, position, quat, velocity, gyro)
    savgol_alpha_x, finite_alpha_x, butter_alpha_x, butter_alpha_y, butter_alpha_z = estimate_angular_acceleration()
    f_body, ddq_pin_joints, alpha = prepare_dynamics_inputs(accel, ddq_joint, butter_alpha_x, butter_alpha_y, butter_alpha_z)
    rotation, a_body = compute_body_acceleration(quat, f_body)
    a_body_filt = filter_body_acceleration(f_body, a_body)
    sample_indices, q_pin_clean, v_pin_clean, a_body_50, ddq_pin_50, gyro_50, alpha_50 = downsample_to_50hz(df_clean, df, q_pin, v_pin, a_body_filt, ddq_pin_joints, gyro, alpha)
    df_cleanest = df_cleanest.iloc[:len(q_pin_clean)].copy()
    imu_pos, adjusted_a_body, a_pin_clean = compute_adjusted_body_acceleration(a_body_50, alpha_50, gyro_50, ddq_pin_50)
    q_pin_rnea, v_pin_rnea, a_pin_rnea = select_rnea_samples(df_cleanest, q_pin_clean, v_pin_clean, a_pin_clean)
    df_cleanest = compute_rnea_forces_moments(go1, q_pin_rnea, v_pin_rnea, a_pin_rnea, df_cleanest)
    data, forward_backward_dist, side_to_side_dist = compute_neutral_foot_positions(go1)
    df_cleanest = compute_zmp_instability(forward_backward_dist, side_to_side_dist, df_cleanest)
    return df_cleanest

In [124]:
generate_corridor_instability()

(18761, 3)
(18761, 3)
(18761, 3)
(3,)
FR_foot: x=0.188, y=-0.127, z=-0.426
FR_foot Center Position:
Front-to-Back Distance (X): 0.188 meters
Side-to-Side Distance  (Y): 0.127 meters

FL_foot: x=0.188, y=0.127, z=-0.426
FL_foot Center Position:
Front-to-Back Distance (X): 0.188 meters
Side-to-Side Distance  (Y): 0.127 meters

RL_foot: x=-0.188, y=0.127, z=-0.426
RL_foot Center Position:
Front-to-Back Distance (X): 0.188 meters
Side-to-Side Distance  (Y): 0.127 meters

RR_foot: x=-0.188, y=-0.127, z=-0.426
RR_foot Center Position:
Front-to-Back Distance (X): 0.188 meters
Side-to-Side Distance  (Y): 0.127 meters

         q0        v0  a0        q1        v1  a1        q2        v2  a2  \
0  0.078358  0.002312  11  0.834751  0.001146   2 -1.609432  0.010879   4   
1  0.078358  0.014819   0  0.834812  0.004150  -9 -1.609432  0.011530  12   
2  0.078358  0.006258 -12  0.834812  0.000916   0 -1.609473 -0.000209  -6   
3  0.078358  0.008513 -13  0.834873  0.012155  19 -1.609473 -0.005182  -1 

,q0,v0,a0,q1,v1,a1,q2,v2,a2,q3,...,Fx,Fy,Fz,Mx,My,X_zmp,Y_zmp,instability_X,instability_Y,instability_index
0,0.078358,0.002312,11,0.834751,0.001146,2,-1.609432,0.010879,4,-0.005995,...,-0.914723,-4.446917,128.976079,1.452577,0.919337,-0.007128,0.011262,0.034253,0.076745,0.076745
1,0.078358,0.014819,0,0.834812,0.004150,-9,-1.609432,0.011530,12,-0.005934,...,0.303498,-2.775923,127.683443,1.603367,0.581201,-0.004552,0.012557,0.021874,0.085570,0.085570
2,0.078358,0.006258,-12,0.834812,0.000916,0,-1.609473,-0.000209,-6,-0.005934,...,-0.863060,-7.152672,128.741808,1.551775,0.098514,-0.000765,0.012053,0.003677,0.082136,0.082136
3,0.078358,0.008513,-13,0.834873,0.012155,19,-1.609473,-0.005182,-1,-0.005995,...,-0.336039,-3.072255,126.333156,1.588503,0.773916,-0.006126,0.012574,0.029438,0.085683,0.085683
4,0.078298,-0.013515,-23,0.834873,0.010916,8,-1.609513,-0.013645,9,-0.005995,...,-1.606415,-5.607324,128.901184,1.368231,1.078267,-0.008365,0.010615,0.040197,0.072331,0.072331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18756,0.065278,-0.210601,49,0.570368,-3.799348,125,-1.638741,9.393224,-434,-0.039482,...,16.819417,-0.326462,125.098147,-1.339377,5.757578,-0.046024,-0.010707,0.221165,0.072958,0.221165
18757,0.073272,0.342480,109,0.613241,3.112039,-255,-1.621019,0.351323,89,-0.027734,...,44.889683,-9.352122,145.969825,-1.103833,-1.366767,0.009363,-0.007562,0.044994,0.051530,0.051530
18758,0.095980,0.817479,-1,0.673251,3.094249,-178,-1.613712,0.041318,-15,-0.025312,...,-1.476671,-4.998272,143.823928,-1.985328,-3.582455,0.024909,-0.013804,0.119695,0.094064,0.119695
18759,0.091196,-0.255277,26,0.735017,3.251912,17,-1.630465,-0.924294,10,-0.043963,...,-64.177870,8.282297,134.841128,0.418982,-7.501009,0.055628,0.003107,0.267316,0.021174,0.267316
